# Structural comparison: Llama-3.2-3B vs Gemma-3-4B

**Zero-GPU** — reads only the HF *configs* (no weights), so it runs in seconds
(needs HF auth for the gated Llama repo). Grounds the §7 architecture discussion
in facts rather than assertion, and shows how MAAT's fixed edit band (layers
7-20) lands on each backbone.

Reports the structural properties that plausibly affect unlearnability:
depth & **edit-band coverage**, hidden/intermediate width, vocab + tied
embeddings, attention heads/GQA, Gemma's **local/global** attention split, and
logit soft-capping / QK-norm (gradient-scale → dim-scoring noise).


In [1]:
!pip install -q transformers


In [2]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.8 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


## HF login (Kaggle)


In [3]:
import os, torch
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
login(hf_token)
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')

GPUs: 2
  GPU 0: Tesla T4  15.6 GB
  GPU 1: Tesla T4  15.6 GB


## Config-reading logic


In [4]:
MID_START, MID_END = 7, 20            # MAAT edit band (inclusive)
LLAMA = "meta-llama/Llama-3.2-3B"
GEMMA = "google/gemma-3-4b-it"

from transformers import AutoConfig

def _text_cfg(cfg):
    # Gemma-3-4b-it is a multimodal config; the LM lives under .text_config
    return getattr(cfg, "text_config", cfg)

def _get(cfg, *names, default=None):
    for n in names:
        v = getattr(cfg, n, None)
        if v is not None:
            return v
    return default

def _global_layers(tc, n_layers):
    # indices of GLOBAL (full) attention layers; None if uniform-global (Llama)
    layer_types = _get(tc, "layer_types")
    if isinstance(layer_types, (list, tuple)):
        return [i for i, t in enumerate(layer_types)
                if "full" in str(t) or "global" in str(t)]
    pat = _get(tc, "sliding_window_pattern")
    if pat and _get(tc, "sliding_window"):
        return [i for i in range(n_layers) if (i + 1) % pat == 0]
    return None

def describe(model_id):
    cfg = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    tc = _text_cfg(cfg)
    n = _get(tc, "num_hidden_layers", default=0)
    band = [l for l in range(n) if MID_START <= l <= MID_END]
    gl = _global_layers(tc, n)
    gl_in_band = [l for l in gl if MID_START <= l <= MID_END] if gl is not None else None
    return {
        "model": model_id.split("/")[-1],
        "num_layers": n,
        "band_coverage_pct": round(100 * len(band) / max(1, n), 1),
        "hidden_size": _get(tc, "hidden_size"),
        "intermediate_size": _get(tc, "intermediate_size"),
        "mlp_ratio": round(_get(tc, "intermediate_size", default=0) /
                           max(1, _get(tc, "hidden_size", default=1)), 2),
        "vocab_size": _get(tc, "vocab_size"),
        "tie_word_embeddings": _get(cfg, "tie_word_embeddings", "tie_embeddings"),
        "num_attention_heads": _get(tc, "num_attention_heads"),
        "num_kv_heads": _get(tc, "num_key_value_heads"),
        "head_dim": _get(tc, "head_dim"),
        "sliding_window": _get(tc, "sliding_window"),
        "sliding_window_pattern": _get(tc, "sliding_window_pattern"),
        "n_global_layers": (len(gl) if gl is not None else "all"),
        "global_layers_in_band": (gl_in_band if gl_in_band is not None else "all"),
        "attn_logit_softcapping": _get(tc, "attn_logit_softcapping"),
        "final_logit_softcapping": _get(tc, "final_logit_softcapping"),
        "query_pre_attn_scalar": _get(tc, "query_pre_attn_scalar"),
        "qk_norm": _get(tc, "use_qk_norm", "query_key_layernorm"),
    }


## Compare


In [5]:
import json, os
rows = []
for mid in (LLAMA, GEMMA):
    try:
        rows.append(describe(mid))
    except Exception as e:
        print(f"[skip {mid}] {e}")

keys = [k for k in rows[0].keys() if k != "model"]
w = max(len(k) for k in keys) + 1
print(f"{'property':<{w}}" + "".join(f"{r['model']:>26}" for r in rows))
print("-" * (w + 26 * len(rows)))
for k in keys:
    print(f"{k:<{w}}" + "".join(f"{str(r.get(k)):>26}" for r in rows))

os.makedirs("results/analysis", exist_ok=True)
json.dump(rows, open("results/analysis/model_structure.json", "w"), indent=2)
print("\nsaved -> results/analysis/model_structure.json")


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

property                              Llama-3.2-3B             gemma-3-4b-it
----------------------------------------------------------------------------
num_layers                                      28                        34
band_coverage_pct                             50.0                      41.2
hidden_size                                   3072                      2560
intermediate_size                             8192                     10240
mlp_ratio                                     2.67                       4.0
vocab_size                                  128256                    262208
tie_word_embeddings                           True                      True
num_attention_heads                             24                         8
num_kv_heads                                     8                         4
head_dim                                       128                       256
sliding_window                                None                      1024

## Reading guide

- **band_coverage_pct** differs → layers 7-20 edit a different fraction of each
  network; if Gemma is deeper, a front-loaded fixed band can miss where facts sit.
- **global_layers_in_band** → Gemma alternates local (sliding-window) and global
  attention; facts needing long-range recall may live in the sparse GLOBAL layers,
  which the uniform edit under-weights. Check how many fall inside 7-20.
- **vocab_size + tie_word_embeddings** → larger vocab / tied embeddings change the
  KL and entropy repair dynamics in Phase 3.
- **soft-capping / qk_norm / query_pre_attn_scalar** → change gradient scale, which
  is what MAAT's top-k gradient dim-scoring (SVD prune, task-vector mask) depends on.

Pair these facts with the measured diagnostics in `cross_architecture_analysis.ipynb`.
